# Hybrid bigvgan vocoder loop training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook tests a hybrid model with a BigVGAN reconstruction path.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:

# =========================
# Installs / imports
# =========================
import os, sys, json, time, math, random, shutil, subprocess
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from IPython.display import Audio, display

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


In [ ]:

# =========================
# Repro / config
# =========================
SEED = 7
DRIVE_ROOT = Path("/content/drive/MyDrive/master")
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
CHECKPOINTS_RUNS_DIR = DRIVE_ROOT / "checkpoints_runs"

# Fresh run only
MODE = "A2MS"
RUN_NAME = "hybrid_bigvgan_vocoder_loop"
RUN_TAG = RUN_NAME + "_" + time.strftime("%Y%m%d_%H%M%S")
CKPT_DIR = CHECKPOINTS_RUNS_DIR / RUN_TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Audio / data
SR = 22050
SEG_S = 1.5
BATCH = 2
NUM_WORKERS = 2
PIN_MEMORY = True
P_SPEECH = 0.7
K_CHOICES = (1, 2)

# BigVGAN / mel frontend
BIGVGAN_MODEL_ID = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False

# BigVGAN-compatible mel params
N_FFT = 1024
HOP = 256
WIN = 1024
N_MELS = 80
FMIN = 0
FMAX = 8000
CENTER_MEL = False

# Complex STFT branch
CPLX_N_FFT = 1024
CPLX_HOP = 256
CPLX_WIN = 1024
CPLX_CENTER = False

# Model sizes
G_BASE = 24
G_GROUPS = 8
CPLX_BASE = 24
FUSION_BASE = 24
D_BASE = 32

# Frequency emphasis
HF_START_FRAC = 0.45
HF_END_GAIN = 8.0
HF_RAMP_POWER = 2.0
MASK_DILATE = 2
TDIFF_MASK_DILATE = 3

# Fusion prior / gate
PRIOR_RADIUS = 3
PRIOR_BOUNDARY_GAIN = 1.5
PRIOR_HF_POWER = 1.25
GATE_TEMP = 2.0

# Optim / schedule
STEPS = 40000
LR_G = 2e-4
LR_D = 1e-4
WEIGHT_DECAY = 1e-4
LOG_EVERY = 100
SAVE_EVERY = 2000

# BigVGAN-in-the-loop losses
USE_ROUNDTRIP = True
ROUNDTRIP_START_STEP = 1
ROUNDTRIP_EVERY = 2
ROUNDTRIP_SUBBATCH = 1

# Adversarial schedule: delayed and light
USE_HF_DISC = True
ADV_VIEW_MODE = "vocoder_roundtrip"   # "vocoder_roundtrip" or "raw_mel"
ADV_START_STEP = 12000
ADV_EVERY = 2
LAMBDA_ADV_MAX = 0.35
LAMBDA_FM = 2.0

# Loss weights
LAMBDA_MAIN_RECON = 15.0
LAMBDA_MAIN_HF = 45.0
LAMBDA_MAIN_TDIFF = 18.0

LAMBDA_MEL_BRANCH = 5.0
LAMBDA_CPLX_BRANCH_MEL = 5.0
LAMBDA_CPLX_COMPLEX = 6.0
LAMBDA_CPLX_LOGMAG = 4.0

LAMBDA_DELTA = 0.05
LAMBDA_GATE = 0.05
LAMBDA_GATE_ALIGN = 0.6

LAMBDA_RT_HF = 12.0
LAMBDA_RT_TDIFF = 5.0

# Optional: make discriminator see only upper part of the spectrogram
HF_D_START_FRAC = 0.50
HF_D_MASK_DILATE = 3

RUN_CONFIG = dict(
    seed=SEED,
    mode=MODE,
    run_name=RUN_NAME,
    run_tag=RUN_TAG,
    fresh_start=True,
    sr=SR,
    segment_s=SEG_S,
    batch=BATCH,
    p_speech=P_SPEECH,
    mel=dict(n_fft=N_FFT, hop=HOP, win=WIN, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, center=CENTER_MEL),
    complex_stft=dict(n_fft=CPLX_N_FFT, hop=CPLX_HOP, win=CPLX_WIN, center=CPLX_CENTER),
    k_choices=list(K_CHOICES),
    g_arch="FreshEndToEndHybrid",
    g_base=G_BASE,
    cplx_base=CPLX_BASE,
    fusion_base=FUSION_BASE,
    d_base=D_BASE,
    adv_view_mode=ADV_VIEW_MODE,
)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("CKPT_DIR:", CKPT_DIR)


In [ ]:
# =========================
# Config sanity check
# =========================
_required = [
    "SR","SEG_S","BATCH","N_FFT","HOP","WIN","N_MELS","FMIN","FMAX",
    "CPLX_N_FFT","CPLX_HOP","CPLX_WIN","HF_START_FRAC","HF_END_GAIN","HF_RAMP_POWER"
]
_missing = [k for k in _required if k not in globals()]
assert not _missing, f"Missing config vars: {_missing}. Run the config cell above first."

print("Config OK:")
print("  SR =", SR)
print("  SEG_S =", SEG_S)
print("  BATCH =", BATCH)
print("  N_MELS =", N_MELS)
print("  HF_START_FRAC =", HF_START_FRAC)
print("  HF_END_GAIN =", HF_END_GAIN)


In [ ]:

# =========================
# BigVGAN repo / imports
# =========================
BIGVGAN_REPO = Path("/content/BigVGAN")
if not BIGVGAN_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/BigVGAN.git", "/content/BigVGAN"], check=True)

if str(BIGVGAN_REPO) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_REPO))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is not None:
        return bigvgan_G

    print("Loading BigVGAN:", BIGVGAN_MODEL_ID)
    bigvgan_G = bigvgan.BigVGAN._from_pretrained(
        model_id=BIGVGAN_MODEL_ID,
        revision="main",
        cache_dir=None,
        force_download=False,
        proxies=None,
        resume_download=False,
        local_files_only=False,
        token=None,
        map_location=str(device),
        strict=False,
        use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
    ).to(device).eval()

    for p in bigvgan_G.parameters():
        p.requires_grad_(False)

    return bigvgan_G

def wav_to_bigvgan_mel(wav_bt: torch.Tensor):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    return bigvgan_mel_spectrogram(
        wav_bt,
        n_fft=N_FFT,
        num_mels=N_MELS,
        sampling_rate=SR,
        hop_size=HOP,
        win_size=WIN,
        fmin=FMIN,
        fmax=FMAX,
        center=CENTER_MEL,
    )

def mel_to_wave_bigvgan(mel_bt: torch.Tensor):
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bt)
    return y.squeeze(1) if (y.ndim == 3 and y.shape[1] == 1) else y

print("BigVGAN repo ready at:", BIGVGAN_REPO)


In [ ]:

# =========================
# Local cache for wav files
# =========================
DRIVE_SPLIT_DIR = SPLIT_DIR
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

def _read_lines(p: Path):
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [ln for ln in lines if ln]

def to_local_path(p: Path) -> Path:
    p = Path(p)
    for root in [DRIVE_ROOT, Path("/content/drive")]:
        try:
            rel = p.relative_to(root)
            return LOCAL_DATA_ROOT / rel
        except Exception:
            pass
    return LOCAL_DATA_ROOT / "flat" / p.name

all_files = []
for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    assert src_list.exists(), f"Missing split list: {src_list}"
    all_files += [Path(x) for x in _read_lines(src_list)]

all_files = sorted(set(all_files))
print("Unique audio files referenced by splits:", len(all_files))

copied = 0
skipped = 0
for src in all_files:
    dst = to_local_path(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        skipped += 1
        continue
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} files (skipped {skipped} already present).")

for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    src_paths = [Path(x) for x in _read_lines(src_list)]
    dst_paths = [str(to_local_path(p)) for p in src_paths]
    (LOCAL_SPLIT_DIR / name).write_text("\n".join(dst_paths) + "\n", encoding="utf-8")

SPLIT_DIR = LOCAL_SPLIT_DIR
RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
RUN_CONFIG["local_data_root"] = str(LOCAL_DATA_ROOT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

print("Using local cached wav paths.")


In [ ]:

# =========================
# Dataset / loaders
# =========================
def read_list(p: Path):
    assert p.exists(), f"Missing list: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr: int, segment_s: float):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(sr * segment_s)
        assert len(self.paths) > 0, f"Empty list: {list_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        w = safe_load_wav_mono(self.paths[idx], self.sr)
        if w.numel() < self.seg_len:
            w = F.pad(w, (0, self.seg_len - w.numel()))
        max_start = w.numel() - self.seg_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        return w[start:start + self.seg_len]

speech_train = SPLIT_DIR / "speech_train.txt"
music_train  = SPLIT_DIR / "music_train.txt"
speech_val   = SPLIT_DIR / "speech_val.txt"
music_val    = SPLIT_DIR / "music_val.txt"

ds_speech = FileListDataset(speech_train, SR, SEG_S)
ds_music  = FileListDataset(music_train,  SR, SEG_S)

dl_speech = DataLoader(
    ds_speech, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
dl_music = DataLoader(
    ds_music, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

def infinite(dl):
    while True:
        for b in dl:
            yield b

it_speech = infinite(dl_speech)
it_music = infinite(dl_music)

def next_mixed_batch():
    b = next(it_speech) if random.random() < P_SPEECH else next(it_music)
    return b.to(device)

print("speech files:", len(ds_speech), "music files:", len(ds_music), "P_SPEECH:", P_SPEECH)


In [ ]:

# =========================
# Helpers: masks, weights, mel/STFT corruption, losses
# =========================
def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

required_cfg = ['N_MELS', 'HF_START_FRAC', 'HF_END_GAIN', 'HF_RAMP_POWER', 'CPLX_N_FFT']
missing_cfg = [k for k in required_cfg if k not in globals()]
assert not missing_cfg, f'Missing config values before helper init: {missing_cfg}. Run the config cell first.'

HF_FREQ_WEIGHTS = build_freq_weights(N_MELS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)
CPLX_FREQ_WEIGHTS = build_freq_weights(CPLX_N_FFT // 2 + 1, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)

def dilate_mask_time(mask: torch.Tensor, radius: int):
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def expand_mask(mask: torch.Tensor, n_bins: int, radius: int = 0):
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, n_bins, -1)

def masked_mean(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    z = x.abs()
    if freq_weights is not None:
        z = z * freq_weights
        denom = (mask_ft * freq_weights).sum()
    else:
        denom = mask_ft.sum()
    return (z * mask_ft).sum() / (denom + eps)

def masked_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    return masked_mean(pred - tgt, m, freq_weights=freq_weights)

def masked_tdiff_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    dp = pred[..., 1:] - pred[..., :-1]
    dt = tgt[..., 1:] - tgt[..., :-1]
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)[..., 1:]
    return masked_mean(dp - dt, m, freq_weights=freq_weights)

def masked_complex_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    diff = torch.complex(pred.real - tgt.real, pred.imag - tgt.imag)
    return masked_mean(diff, m, freq_weights=freq_weights)

def masked_logmag_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    pm = torch.log1p(pred.abs())
    tm = torch.log1p(tgt.abs())
    return masked_l1(pm, tm, mask, freq_weights=freq_weights, mask_dilate=mask_dilate)

def linear_time_inpaint_mel(mel: torch.Tensor, k: int, offset: int):
    B, F, T = mel.shape
    step = k + 1
    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, 1, T), device=mel.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, :, t] = 1.0
    return mel_interp, mask

def stft_complex(wav_bt: torch.Tensor):
    window = torch.hann_window(CPLX_WIN, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        return_complex=True,
    )

def istft_complex(X_bft: torch.Tensor, length: int):
    window = torch.hann_window(CPLX_WIN, device=X_bft.device)
    return torch.istft(
        X_bft,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        length=length,
    )

def linear_time_inpaint_complex(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def make_shared_pair(wav_bt: torch.Tensor, k_choices=(1, 2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0

    mel_real = wav_to_bigvgan_mel(wav_bt)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_real = stft_complex(wav_bt)
    X_interp, mask_cplx = linear_time_inpaint_complex(X_real, k=k, offset=offset)

    return {
        "wav": wav_bt,
        "mel_real": mel_real,
        "mel_interp": mel_interp,
        "mask_mel": mask_mel,
        "X_real": X_real,
        "X_interp": X_interp,
        "mask_cplx": mask_cplx,
        "k": k,
        "offset": offset,
    }

def make_hf_prior(mask_mel: torch.Tensor, n_mels: int, radius: int = PRIOR_RADIUS):
    base = expand_mask(mask_mel, n_mels, radius=0)
    wide = expand_mask(mask_mel, n_mels, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    hf = HF_FREQ_WEIGHTS.expand(wide.shape[0], -1, wide.shape[-1])
    hf_norm = ((hf - 1.0) / max(1e-6, (HF_END_GAIN - 1.0))).clamp(0.0, 1.0)
    hf_shape = hf_norm ** PRIOR_HF_POWER

    region = (base + PRIOR_BOUNDARY_GAIN * boundary).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)
    return (region * hf_shape).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)

def make_gate_target(mel_main_hat: torch.Tensor, mel_complex_hat: torch.Tensor, mel_real: torch.Tensor, prior: torch.Tensor):
    err_main = (mel_main_hat - mel_real).abs()
    err_cplx = (mel_complex_hat - mel_real).abs()
    gain = (err_main - err_cplx).clamp_min(0.0) * prior
    den = gain.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)
    target = (gain / den).clamp(0.0, 1.0)
    return target ** 0.75

def make_adv_focus_view(mel_bt: torch.Tensor, mask_mel: torch.Tensor):
    start_bin = int(round(HF_D_START_FRAC * mel_bt.shape[1]))
    mel_hf = mel_bt[:, start_bin:, :]
    mask_hf = expand_mask(mask_mel, mel_bt.shape[1], radius=HF_D_MASK_DILATE)[:, start_bin:, :]
    return mel_hf * mask_hf, mask_hf

def get_roundtrip_views(mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor, with_grad_fake: bool = True):
    sb = min(ROUNDTRIP_SUBBATCH, mel_hat.shape[0])
    mel_hat_sb = mel_hat[:sb]
    mel_real_sb = mel_real[:sb]
    mask_sb = mask_mel[:sb]

    if ADV_VIEW_MODE == "raw_mel":
        mel_view_hat = mel_hat_sb
        mel_view_real = mel_real_sb
    else:
        if with_grad_fake:
            wav_hat = mel_to_wave_bigvgan(mel_hat_sb)
        else:
            with torch.no_grad():
                wav_hat = mel_to_wave_bigvgan(mel_hat_sb)

        with torch.no_grad():
            wav_real = mel_to_wave_bigvgan(mel_real_sb)

        mel_view_hat = wav_to_bigvgan_mel(wav_hat)
        mel_view_real = wav_to_bigvgan_mel(wav_real)

    T = min(mel_view_hat.shape[-1], mel_view_real.shape[-1], mask_sb.shape[-1])
    mel_view_hat = mel_view_hat[..., :T]
    mel_view_real = mel_view_real[..., :T]
    mask_sb = mask_sb[..., :T]

    return mel_view_hat, mel_view_real, mask_sb


In [ ]:

# =========================
# Models: mel branch, complex branch, fusion gate, HF discriminator
# =========================
def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            act_layer,
        )
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, groups=groups, act=act),
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2,2), groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            act_layer,
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups, act=act)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)

class MelBranchUNet2D(nn.Module):
    def __init__(self, n_mels, base=24, use_mask=True, groups=8):
        super().__init__()
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)
        self.enc1 = DoubleConv(c_in, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        delta = self.out_conv(u0).squeeze(1)
        return delta

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ComplexBranchUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

class FusionUNet2D(nn.Module):
    def __init__(self, in_ch=5, base=24, groups=8):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.gate_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.delta_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
        nn.init.zeros_(self.delta_head.weight); nn.init.zeros_(self.delta_head.bias)

    def forward(self, mel_interp, mel_main, mel_complex, prior, mask):
        x = torch.stack([
            mel_interp,
            mel_main,
            mel_complex,
            prior,
            mask.expand(-1, mel_interp.shape[1], -1),
        ], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        gate_logits = self.gate_head(u0).squeeze(1)
        gate = torch.sigmoid(GATE_TEMP * gate_logits)
        delta = self.delta_head(u0).squeeze(1)
        return gate, delta

class HFDisc(nn.Module):
    def __init__(self, in_ch=2, base=32):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(spectral_norm(nn.Conv2d(in_ch, base, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base, base*2, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*2, base*4, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*4, base*4, 3, stride=1, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.out = spectral_norm(nn.Conv2d(base*4, 1, 3, stride=1, padding=1))

    def forward(self, x):
        feats = []
        h = x
        for blk in self.blocks:
            h = blk(h)
            feats.append(h)
        logit = self.out(h)
        return logit, feats

MelG = MelBranchUNet2D(n_mels=N_MELS, base=G_BASE, use_mask=True, groups=G_GROUPS).to(device)
CplxG = ComplexBranchUNet2D(in_ch=3, out_ch=2, base=CPLX_BASE, groups=G_GROUPS).to(device)
FuseG = FusionUNet2D(in_ch=5, base=FUSION_BASE, groups=G_GROUPS).to(device)
Dhf = HFDisc(in_ch=2, base=D_BASE).to(device)

print("MelG params (M):", sum(p.numel() for p in MelG.parameters()) / 1e6)
print("CplxG params (M):", sum(p.numel() for p in CplxG.parameters()) / 1e6)
print("FuseG params (M):", sum(p.numel() for p in FuseG.parameters()) / 1e6)
print("Dhf params (M):", sum(p.numel() for p in Dhf.parameters()) / 1e6)


In [ ]:

# =========================
# Optimizers / checkpoint helpers
# =========================
g_params = list(MelG.parameters()) + list(CplxG.parameters()) + list(FuseG.parameters())
opt_g = torch.optim.AdamW(g_params, lr=LR_G, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)
opt_d = torch.optim.AdamW(Dhf.parameters(), lr=LR_D, betas=(0.8, 0.99), weight_decay=WEIGHT_DECAY)

def save_ckpt(step: int):
    obj = {
        "step": step,
        "mode": MODE,
        "run_config": RUN_CONFIG,
        "MelG": MelG.state_dict(),
        "CplxG": CplxG.state_dict(),
        "FuseG": FuseG.state_dict(),
        "Dhf": Dhf.state_dict(),
        "opt_g": opt_g.state_dict(),
        "opt_d": opt_d.state_dict(),
        "hf_freq_weights": HF_FREQ_WEIGHTS.detach().cpu(),
        "cplx_freq_weights": CPLX_FREQ_WEIGHTS.detach().cpu(),
    }
    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)

start_step = 0
print("Starting fresh from scratch.")


In [ ]:

# =========================
# Forward helpers
# =========================
def mel_branch_forward(mel_interp: torch.Tensor, mask_mel: torch.Tensor):
    delta = MelG(mel_interp, mask_mel[:, :, 0, :])
    mask_ft = mask_mel[:, 0].expand(-1, mel_interp.shape[1], -1)
    return mel_interp + delta * mask_ft, delta

def complex_branch_forward(X_interp: torch.Tensor, mask_cplx: torch.Tensor, wav_len: int):
    x_in = pack_complex_input(X_interp, mask_cplx)
    delta = CplxG(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_hat = istft_complex(X_hat, length=wav_len)
    mel_hat = wav_to_bigvgan_mel(wav_hat)
    return X_hat, wav_hat, mel_hat, dX

def fusion_forward(mel_interp, mel_main_hat, mel_complex_hat, mel_real, mask_mel):
    prior = make_hf_prior(mask_mel, n_mels=mel_main_hat.shape[1], radius=PRIOR_RADIUS)
    gate_target = make_gate_target(mel_main_hat.detach(), mel_complex_hat.detach(), mel_real=mel_real.detach(), prior=prior.detach())
    gate, delta = FuseG(
        mel_interp=mel_interp,
        mel_main=mel_main_hat,
        mel_complex=mel_complex_hat,
        prior=prior,
        mask=mask_mel[:, :, 0, :],
    )
    borrowed = gate * (mel_complex_hat - mel_main_hat)
    mel_fused = mel_main_hat + prior * (borrowed + delta)
    return mel_fused, gate, delta, borrowed, prior, gate_target

def disc_input_from_view(mel_view: torch.Tensor, mask_view: torch.Tensor):
    mel_hf, mask_hf = make_adv_focus_view(mel_view, mask_view)
    return torch.stack([mel_hf, mask_hf], dim=1)

def adv_weight(step: int):
    if step < ADV_START_STEP:
        return 0.0
    return LAMBDA_ADV_MAX


In [ ]:

# =========================
# Train
# =========================
MelG.train()
CplxG.train()
FuseG.train()
Dhf.train()

log_path = CKPT_DIR / "train_log.csv"
if not log_path.exists():
    log_path.write_text(
        "step,loss_total,loss_main_recon,loss_main_hf,loss_main_tdiff,"
        "loss_mel_branch,loss_cplx_branch_mel,loss_cplx_complex,loss_cplx_logmag,"
        "loss_delta,loss_gate,loss_gate_align,loss_rt_hf,loss_rt_tdiff,"
        "loss_adv,loss_fm,adv_w,lossD,dr_hf,df_hf,"
        "base_hf,ref_hf,imp_hf_pct,avg_gate,borrow_hf,k,offset,used_roundtrip,minutes\n",
        encoding="utf-8",
    )

t0 = time.time()

for step in range(start_step + 1, start_step + STEPS + 1):
    wav = next_mixed_batch()
    pair = make_shared_pair(wav, k_choices=K_CHOICES, randomize_offset=True)

    wav_real = pair["wav"]
    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]

    X_real = pair["X_real"]
    X_interp = pair["X_interp"]
    mask_cplx = pair["mask_cplx"]

    # branch forwards
    mel_main_hat, mel_delta = mel_branch_forward(mel_interp, mask_mel)
    X_hat, wav_cplx_hat, mel_cplx_hat, dX = complex_branch_forward(X_interp, mask_cplx, wav_len=wav_real.shape[-1])

    # align mel time dimensions
    Tm = min(mel_real.shape[-1], mel_interp.shape[-1], mel_main_hat.shape[-1], mel_cplx_hat.shape[-1], mask_mel.shape[-1])
    mel_real = mel_real[..., :Tm]
    mel_interp = mel_interp[..., :Tm]
    mel_main_hat = mel_main_hat[..., :Tm]
    mel_cplx_hat = mel_cplx_hat[..., :Tm]
    mask_mel = mask_mel[..., :Tm]

    mel_fused, gate, delta, borrowed, prior, gate_target = fusion_forward(mel_interp, mel_main_hat, mel_cplx_hat, mel_real, mask_mel)

    # main fused losses
    loss_main_recon = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
    loss_main_hf    = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_main_tdiff = masked_tdiff_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

    # keep branches individually useful
    loss_mel_branch = masked_l1(mel_main_hat, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
    loss_cplx_branch_mel = masked_l1(mel_cplx_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)

    loss_cplx_complex = masked_complex_l1(X_hat, X_real, mask_cplx, freq_weights=CPLX_FREQ_WEIGHTS, mask_dilate=1)
    loss_cplx_logmag  = masked_logmag_l1(X_hat, X_real, mask_cplx, freq_weights=CPLX_FREQ_WEIGHTS, mask_dilate=1)

    # fusion regularizers
    loss_delta = ((delta.abs() * prior).sum() / (prior.sum() + 1e-8))
    loss_gate = ((gate.abs() * prior).sum() / (prior.sum() + 1e-8))
    loss_gate_align = (((gate - gate_target).abs() * prior).sum() / (prior.sum() + 1e-8))

    # roundtrip + post-vocoder views
    do_rt = USE_ROUNDTRIP and (step >= ROUNDTRIP_START_STEP) and (step % ROUNDTRIP_EVERY == 0)
    if do_rt:
        mel_view_hat, mel_view_real, mask_view = get_roundtrip_views(mel_fused, mel_real, mask_mel, with_grad_fake=True)
        loss_rt_hf = masked_l1(mel_view_hat, mel_view_real, mask_view, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        loss_rt_tdiff = masked_tdiff_l1(mel_view_hat, mel_view_real, mask_view, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)
    else:
        mel_view_hat = mel_view_real = mask_view = None
        loss_rt_hf = torch.zeros((), device=device)
        loss_rt_tdiff = torch.zeros((), device=device)

    # discriminator step
    use_adv = USE_HF_DISC and do_rt and (step >= ADV_START_STEP) and (step % ADV_EVERY == 0)
    adv_w = adv_weight(step)
    lossD = torch.zeros((), device=device)
    dr_hf = 0.0
    df_hf = 0.0

    if use_adv:
        with torch.no_grad():
            inp_real = disc_input_from_view(mel_view_real.detach(), mask_view.detach())
            inp_fake = disc_input_from_view(mel_view_hat.detach(), mask_view.detach())

        pred_r, _ = Dhf(inp_real)
        pred_f, _ = Dhf(inp_fake)

        lossD_real = ((pred_r - 1.0) ** 2).mean()
        lossD_fake = (pred_f ** 2).mean()
        lossD = 0.5 * (lossD_real + lossD_fake)

        opt_d.zero_grad(set_to_none=True)
        lossD.backward()
        torch.nn.utils.clip_grad_norm_(Dhf.parameters(), 5.0)
        opt_d.step()

        dr_hf = float(pred_r.mean().item())
        df_hf = float(pred_f.mean().item())

    # generator adversarial step
    loss_adv = torch.zeros((), device=device)
    loss_fm = torch.zeros((), device=device)
    if use_adv:
        inp_fake_g = disc_input_from_view(mel_view_hat, mask_view)
        with torch.no_grad():
            inp_real_g = disc_input_from_view(mel_view_real, mask_view)

        pred_fake_g, feats_fake = Dhf(inp_fake_g)
        pred_real_g, feats_real = Dhf(inp_real_g)
        loss_adv = ((pred_fake_g - 1.0) ** 2).mean()
        loss_fm = sum(F.l1_loss(ff, fr.detach()) for ff, fr in zip(feats_fake, feats_real))

    loss_total = (
        LAMBDA_MAIN_RECON * loss_main_recon
        + LAMBDA_MAIN_HF * loss_main_hf
        + LAMBDA_MAIN_TDIFF * loss_main_tdiff
        + LAMBDA_MEL_BRANCH * loss_mel_branch
        + LAMBDA_CPLX_BRANCH_MEL * loss_cplx_branch_mel
        + LAMBDA_CPLX_COMPLEX * loss_cplx_complex
        + LAMBDA_CPLX_LOGMAG * loss_cplx_logmag
        + LAMBDA_DELTA * loss_delta
        + LAMBDA_GATE * loss_gate
        + LAMBDA_GATE_ALIGN * loss_gate_align
        + LAMBDA_RT_HF * loss_rt_hf
        + LAMBDA_RT_TDIFF * loss_rt_tdiff
        + adv_w * loss_adv
        + LAMBDA_FM * loss_fm
    )

    opt_g.zero_grad(set_to_none=True)
    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(g_params, 5.0)
    opt_g.step()

    with torch.no_grad():
        base_hf = masked_l1(mel_main_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        ref_hf  = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        imp_hf_pct = 100.0 * (base_hf.item() - ref_hf.item()) / max(1e-8, base_hf.item())
        avg_gate = float((gate * prior).sum().item() / (prior.sum().item() + 1e-8))
        borrow_hf = float(((borrowed.abs() * prior).sum() / (prior.sum() + 1e-8)).item())

    if (step % LOG_EVERY == 0) or (step == start_step + 1):
        mins = (time.time() - t0) / 60.0
        row = [
            step,
            float(loss_total.item()),
            float(loss_main_recon.item()),
            float(loss_main_hf.item()),
            float(loss_main_tdiff.item()),
            float(loss_mel_branch.item()),
            float(loss_cplx_branch_mel.item()),
            float(loss_cplx_complex.item()),
            float(loss_cplx_logmag.item()),
            float(loss_delta.item()),
            float(loss_gate.item()),
            float(loss_gate_align.item()),
            float(loss_rt_hf.item()),
            float(loss_rt_tdiff.item()),
            float(loss_adv.item()),
            float(loss_fm.item()),
            float(adv_w),
            float(lossD.item()),
            float(dr_hf),
            float(df_hf),
            float(base_hf.item()),
            float(ref_hf.item()),
            float(imp_hf_pct),
            float(avg_gate),
            float(borrow_hf),
            int(pair["k"]),
            int(pair["offset"]),
            int(do_rt),
            float(mins),
        ]
        with log_path.open("a", encoding="utf-8") as f:
            f.write(",".join(map(str, row)) + "\n")

        print(
            f"step {step:7d} | "
            f"loss {loss_total.item():.4f} | "
            f"main_hf {loss_main_hf.item():.4f} | "
            f"rt_hf {loss_rt_hf.item():.4f} | "
            f"adv {loss_adv.item():.4f} fm {loss_fm.item():.4f} | "
            f"D {lossD.item():.4f} | "
            f"gate {avg_gate:.3f} | borrow_hf {borrow_hf:.4f} | "
            f"imp_hf {imp_hf_pct:.2f}% | "
            f"k {pair['k']} off {pair['offset']}"
        )

    if (step % SAVE_EVERY == 0) or (step == start_step + STEPS):
        save_ckpt(step)

print("done. CKPT_DIR:", CKPT_DIR)


In [ ]:

# =========================
# Training curves
# =========================
log_df = pd.read_csv(CKPT_DIR / "train_log.csv")
display(log_df.tail())

plt.figure(figsize=(10, 4))
plt.plot(log_df["step"], log_df["loss_total"], label="loss_total")
plt.plot(log_df["step"], log_df["loss_main_hf"], label="loss_main_hf")
plt.plot(log_df["step"], log_df["loss_rt_hf"], label="loss_rt_hf")
plt.plot(log_df["step"], log_df["loss_adv"], label="loss_adv")
plt.plot(log_df["step"], log_df["loss_fm"], label="loss_fm")
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": main losses")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(log_df["step"], log_df["imp_hf_pct"], label="imp_hf_pct")
plt.axhline(0.0, ls="--", c="k", lw=1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": HF improvement over mel branch")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(log_df["step"], log_df["avg_gate"], label="avg_gate")
plt.plot(log_df["step"], log_df["borrow_hf"], label="borrow_hf")
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": gate / borrowing")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(log_df["step"], log_df["dr_hf"], label="dr_hf")
plt.plot(log_df["step"], log_df["df_hf"], label="df_hf")
plt.legend()
plt.grid(True, alpha=0.3)
plt.title(RUN_TAG + ": HF discriminator outputs")
plt.show()


In [ ]:

# =========================
# Quick sanity listen
# =========================
@torch.no_grad()
def eval_one_file(path: Path, k=1, offset=0, do_listen=True):
    MelG.eval(); CplxG.eval(); FuseG.eval()
    w = safe_load_wav_mono(path, SR)[: int(SR * SEG_S)].to(device)
    wb = w.unsqueeze(0)

    mel_real = wav_to_bigvgan_mel(wb)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_real = stft_complex(wb)
    X_interp, mask_cplx = linear_time_inpaint_complex(X_real, k=k, offset=offset)

    mel_main_hat, _ = mel_branch_forward(mel_interp, mask_mel)
    X_hat, wav_cplx_hat, mel_cplx_hat, _ = complex_branch_forward(X_interp, mask_cplx, wav_len=w.shape[-1])

    T = min(mel_real.shape[-1], mel_interp.shape[-1], mel_main_hat.shape[-1], mel_cplx_hat.shape[-1], mask_mel.shape[-1])
    mel_real = mel_real[..., :T]
    mel_interp = mel_interp[..., :T]
    mel_main_hat = mel_main_hat[..., :T]
    mel_cplx_hat = mel_cplx_hat[..., :T]
    mask_mel = mask_mel[..., :T]

    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=PRIOR_RADIUS)
    gate_target = make_gate_target(mel_main_hat, mel_cplx_hat, mel_real, prior)
    gate, delta = FuseG(mel_interp, mel_main_hat, mel_cplx_hat, prior, mask_mel[:, :, 0, :])
    mel_fused = mel_main_hat + prior * (gate * (mel_cplx_hat - mel_main_hat) + delta)

    wav_interp = mel_to_wave_bigvgan(mel_interp)
    wav_main = mel_to_wave_bigvgan(mel_main_hat)
    wav_fused = mel_to_wave_bigvgan(mel_fused)

    base_hf = masked_l1(mel_main_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    ref_hf  = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    print(f"{path.name} | k={k} off={offset} | HF {base_hf:.6f} -> {ref_hf:.6f}")

    if do_listen:
        print("Original")
        display(Audio(w.detach().cpu().numpy(), rate=SR))
        print("Interpolated -> BigVGAN")
        display(Audio(wav_interp[0].detach().cpu().numpy(), rate=SR))
        print("Mel branch -> BigVGAN")
        display(Audio(wav_main[0].detach().cpu().numpy(), rate=SR))
        print("Complex branch -> ISTFT")
        display(Audio(wav_cplx_hat[0].detach().cpu().numpy(), rate=SR))
        print("Hybrid fused -> BigVGAN")
        display(Audio(wav_fused[0].detach().cpu().numpy(), rate=SR))

    MelG.train(); CplxG.train(); FuseG.train()
    return {
        "wav_real": w.detach().cpu(),
        "wav_interp": wav_interp[0].detach().cpu(),
        "wav_main": wav_main[0].detach().cpu(),
        "wav_cplx": wav_cplx_hat[0].detach().cpu(),
        "wav_fused": wav_fused[0].detach().cpu(),
        "mel_real": mel_real[0].detach().cpu(),
        "mel_main": mel_main_hat[0].detach().cpu(),
        "mel_cplx": mel_cplx_hat[0].detach().cpu(),
        "mel_fused": mel_fused[0].detach().cpu(),
        "gate": gate[0].detach().cpu(),
        "prior": prior[0].detach().cpu(),
        "gate_target": gate_target[0].detach().cpu(),
    }

speech_path = read_list(speech_val)[0]
_ = eval_one_file(speech_path, k=1, offset=0, do_listen=False)
